<a href="https://colab.research.google.com/github/ndix212-bot/ecommerce-sales-data-cleaner/blob/main/corporate_finance_etl_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Create messy Income Statement data
income_data = {
    'Fiscal_Period': ['Q1-2025', 'Q2 2025', 'Q3_2025', 'Q4-2025', 'Q1-2026', None, 'Q2-2026'],
    'Revenue': ['$5,200,000', '€4,800,000', '$5,500,000*', '$5,900,000', '€5,100,000', '$4,200,000', '$6,100,000_est'],
    'OPEX': ['$3,100,000', '€2,900,000', '3200000', '$3,400,000', '€3,000,000', None, '$3,600,000'],
    'Net_Income': ['$2,100,000', '€1,900,000', '$2,300,000', '$2,500,000', '€2,100,000', '  ', '$2,500,000']
}

# Create messy Transaction Ledger data
ledger_data = {
    'Txn_ID': ['TXN001', 'TXN002', 'TXN003', 'TXN003', 'TXN004', 'TXN005'],
    'Account_Type': ['Revenue  ', ' OPEX', 'OPEX', 'OPEX', 'Revenue', 'OPEX'],
    'Amount': [25000.00, -1200.50, -4500.00, -4500.00, 18500.00, np.nan],
    'Currency': ['USD', 'EUR', 'USD', 'USD', 'USD', 'USD']
}

df_income = pd.DataFrame(income_data)
df_ledger = pd.DataFrame(ledger_data)

# Save to a multi-tab Excel file
with pd.ExcelWriter('raw_target_financials.xlsx') as writer:
    df_income.to_excel(writer, sheet_name='Income_Statement', index=False)
    df_ledger.to_excel(writer, sheet_name='Transaction_Ledger', index=False)

print("Messy financial data file 'raw_target_financials.xlsx' successfully generated!")

Messy financial data file 'raw_target_financials.xlsx' successfully generated!


In [2]:
import pandas as pd
import numpy as np

# 1. Load the data
file_path = 'raw_target_financials.xlsx'
df_income = pd.read_excel(file_path, sheet_name='Income_Statement')
df_ledger = pd.read_excel(file_path, sheet_name='Transaction_Ledger')

print("--- Starting Financial Data Cleaning Pipeline ---\n")

# ==========================================
# CLEANING SHEET 1: INCOME STATEMENT
# ==========================================

# Drop rows where critical time-period data is completely missing
df_income.dropna(subset=['Fiscal_Period'], inplace=True)

# Standardize Fiscal Period formatting (e.g., "Q2 2025" or "Q3_2025" -> "Q2-2025")
df_income['Fiscal_Period'] = df_income['Fiscal_Period'].str.replace(' ', '-').str.replace('_', '-')

# Function to normalize and clean corporate financial figures
def clean_financial_currency(val, exchange_rate=1.10):
    if pd.isna(val) or str(val).strip() == '':
        return 0.0

    val_str = str(val).upper().strip()

    # Check currency flag
    is_eur = '€' in val_str or 'EUR' in val_str

    # Remove currency symbols and footnote markers (*, _est, commas)
    for char in ['$', '€', ',', '*', '_EST', ' ']:
        val_str = val_str.replace(char, '')

    try:
        numeric_val = float(val_str)
        # Convert EUR to USD if flagged
        if is_eur:
            numeric_val = numeric_val * exchange_rate
        return round(numeric_val, 2)
    except ValueError:
        return 0.0

# Apply corporate financial text parsing
financial_cols = ['Revenue', 'OPEX', 'Net_Income']
for col in financial_cols:
    df_income[col] = df_income[col].apply(clean_financial_currency)

# Calculate calculated field audit verify column: EBITDA/Net Income Verification
# Formula: Revenue - OPEX = Calculated_Net_Income
df_income['Calculated_Net_Income'] = df_income['Revenue'] - df_income['OPEX']
df_income['Audit_Match'] = df_income['Net_Income'] == df_income['Calculated_Net_Income']


# ==========================================
# CLEANING SHEET 2: TRANSACTION LEDGER
# ==========================================

# Strip trailing/leading whitespaces from categorical accounting strings
df_ledger['Account_Type'] = df_ledger['Account_Type'].str.strip()
df_ledger['Txn_ID'] = df_ledger['Txn_ID'].str.strip()

# Deduplicate identical internal transaction ledger entries
duplicate_count = df_ledger.duplicated(subset=['Txn_ID']).sum()
print(f"Identified and removed {duplicate_count} duplicate ledger transactions.")
df_ledger.drop_duplicates(subset=['Txn_ID'], keep='first', inplace=True)

# Handle missing transaction values by filling with the median ledger amount
median_amount = df_ledger['Amount'].median()
df_ledger['Amount'].fillna(median_amount, inplace=True)

print("\n--- Cleaning Pipeline Complete! ---")

# Preview the clean dataframes
print("\nCleaned Income Statement:")
print(df_income)

print("\nCleaned Transaction Ledger:")
print(df_ledger)

# Save cleaned output
with pd.ExcelWriter('cleaned_target_financials.xlsx') as writer:
    df_income.to_excel(writer, sheet_name='Clean_Income_Statement', index=False)
    df_ledger.to_excel(writer, sheet_name='Clean_Transaction_Ledger', index=False)

--- Starting Financial Data Cleaning Pipeline ---

Identified and removed 1 duplicate ledger transactions.

--- Cleaning Pipeline Complete! ---

Cleaned Income Statement:
  Fiscal_Period    Revenue       OPEX  Net_Income  Calculated_Net_Income  \
0       Q1-2025  5200000.0  3100000.0   2100000.0              2100000.0   
1       Q2-2025  5280000.0  3190000.0   2090000.0              2090000.0   
2       Q3-2025  5500000.0  3200000.0   2300000.0              2300000.0   
3       Q4-2025  5900000.0  3400000.0   2500000.0              2500000.0   
4       Q1-2026  5610000.0  3300000.0   2310000.0              2310000.0   
6       Q2-2026  6100000.0  3600000.0   2500000.0              2500000.0   

   Audit_Match  
0         True  
1         True  
2         True  
3         True  
4         True  
6         True  

Cleaned Transaction Ledger:
   Txn_ID Account_Type    Amount Currency
0  TXN001      Revenue  25000.00      USD
1  TXN002         OPEX  -1200.50      EUR
2  TXN003         OPEX

/tmp/ipykernel_14437/833876520.py:70: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_ledger['Amount'].fillna(median_amount, inplace=True)


In [3]:
import sqlite3
import pandas as pd

# 1. Connect to an SQLite Database file (it will create it automatically)
conn = sqlite3.connect('corporate_financials.db')
cursor = conn.cursor()

print("--- Step 3: Starting Audit & Database Load ---")

# 2. Check for data discrepancies (Auditing)
discrepancies = df_income[df_income['Audit_Match'] == False]

if not discrepancies.empty:
    print(f"\n⚠️ AUDIT ALERT: Found {len(discrepancies)} row(s) where reported Net Income does not match Revenue - OPEX!")
    print(discrepancies[['Fiscal_Period', 'Revenue', 'OPEX', 'Net_Income', 'Calculated_Net_Income']])
else:
    print("\n✅ AUDIT PASSED: All income statement rows perfectly recalculate.")

# 3. Write data frames to SQL Tables
df_income.to_sql('income_statement', conn, if_exists='replace', index=False)
df_ledger.to_sql('transaction_ledger', conn, if_exists='replace', index=False)
print("\n📦 Clean datasets successfully written to SQLite tables: 'income_statement' and 'transaction_ledger'.")

# 4. SQL Verification: Run a test query to pull high-value transactions
print("\n🔍 Executing SQL Verification Query (Pulling ledger transactions)...")
query = """
SELECT Txn_ID, Account_Type, Amount, Currency
FROM transaction_ledger
WHERE Account_Type = 'Revenue'
ORDER BY Amount DESC;
"""
sql_result = pd.read_sql_query(query, conn)
print(sql_result)

# Close connection
conn.close()
print("\n--- Process Complete! Your clean data room is locked in. ---")

--- Step 3: Starting Audit & Database Load ---

✅ AUDIT PASSED: All income statement rows perfectly recalculate.

📦 Clean datasets successfully written to SQLite tables: 'income_statement' and 'transaction_ledger'.

🔍 Executing SQL Verification Query (Pulling ledger transactions)...
   Txn_ID Account_Type   Amount Currency
0  TXN001      Revenue  25000.0      USD
1  TXN004      Revenue  18500.0      USD

--- Process Complete! Your clean data room is locked in. ---
